In [2]:
import pandas as pd
import numpy as np
import random


def gerar_dataset_quality(n_linhas=500, proporcao_defeitos=0.5, seed=42):
    np.random.seed(seed)
    random.seed(seed)

    nomes = [
        "Ana", "Bruno", "Carlos", "Daniela", "Eduardo",
        "Fernanda", "Gabriel", "Helena", "Igor", "Juliana"
    ]
    departamentos = ["Financeiro", "TI", "RH", "Marketing", "Operações"]

    # 1) Base limpa
    df = pd.DataFrame({
        "id_funcionario": range(1, n_linhas + 1),
        "nome": np.random.choice(nomes, n_linhas),
        "idade": np.random.randint(22, 60, n_linhas),
        "salario": np.random.randint(3000, 15000, n_linhas),
        "departamento": np.random.choice(departamentos, n_linhas),
        "tempo_empresa_anos": np.random.randint(0, 20, n_linhas),
        "status_constante": ["ATIVO"] * n_linhas
    })

    # IMPORTANTE:
    # convertemos colunas que poderão receber valores "quebrados"
    df["salario"] = df["salario"].astype(object)
    df["tempo_empresa_anos"] = df["tempo_empresa_anos"].astype(object)

    # Garante pelo menos 50% limpo
    proporcao_defeitos = min(proporcao_defeitos, 0.5)
    n_defeitos = int(n_linhas * proporcao_defeitos)

    linhas_defeituosas = np.random.choice(df.index, n_defeitos, replace=False)
    tipos_defeito = ["missing", "outlier", "string_numerico"]
    escolhas = np.random.choice(tipos_defeito, n_defeitos)

    # 2) Injeção de defeitos
    for idx, tipo in zip(linhas_defeituosas, escolhas):
        if tipo == "missing":
            coluna = random.choice(["idade", "salario", "departamento"])
            df.loc[idx, coluna] = np.nan

        elif tipo == "outlier":
            coluna = random.choice(["salario", "idade"])
            if coluna == "salario":
                df.loc[idx, "salario"] = random.choice([999999, -3000])
            else:
                df.loc[idx, "idade"] = random.choice([5, 130])

        elif tipo == "string_numerico":
            coluna = random.choice(["salario", "tempo_empresa_anos"])
            df.loc[idx, coluna] = random.choice(["erro", "desconhecido", "abc"])

    # 3) Duplicatas
    n_dup = max(1, int(n_linhas * 0.05))
    duplicatas = df.sample(n=n_dup, random_state=seed)
    df = pd.concat([df, duplicatas], ignore_index=True)

    return df

In [3]:
df = gerar_dataset_quality(1000)

df.head()

,id_funcionario,nome,idade,salario,departamento,tempo_empresa_anos,status_constante
0,1,Gabriel,54.0,6083,Marketing,8,ATIVO
1,2,Daniela,33.0,13123,NaN,12,ATIVO
2,3,Helena,57.0,999999,Financeiro,13,ATIVO
3,4,Eduardo,25.0,10123,Operações,9,ATIVO
4,5,Gabriel,NaN,13754,TI,5,ATIVO


In [4]:
df.to_csv("../Data/dataset_data_quality.csv", index=False)